# What is Locust?

Locust is:
* an easy-to-use
* event-based
* scriptable
* scalable
* extendable, and
* distributed performance testing library/framework.



# FirstScript.py

```python
from locust import User, between, task


class MyUser(User):
    wait_time = between(1,2)

    @task
    def my_task1(self):
        print("MyUser my_task1 executing")

    @task
    def my_task2(self):
        print("MyUser my_task2 executing")
```

**Run**: `locust -f FirstScript.py`

```
locust -f FirstScript.py
[2025-06-29 00:28:18,537] EPINPUNW0254/INFO/locust.main: Starting Locust 2.37.11
[2025-06-29 00:28:18,538] EPINPUNW0254/INFO/locust.main: Starting web interface at http://localhost:8089, press enter to open your default browser.
[2025-06-29 00:28:55,959] EPINPUNW0254/INFO/locust.runners: Ramping to 1 users at a rate of 1.00 per second
[2025-06-29 00:28:55,962] EPINPUNW0254/INFO/locust.runners: All users spawned: {"MyUser": 1} (1 total users)
MyUser my_task2 executing
MyUser my_task2 executing
MyUser my_task2 executing
MyUser my_task1 executing
MyUser my_task1 executing
MyUser my_task2 executing
MyUser my_task2 executing
MyUser my_task2 executing
Traceback (most recent call last):
  File "D:\practice_workspace\python_space\locust-python\.venv\Lib\site-packages\gevent\_ffi\loop.py", line 270, in python_check_callback
    def python_check_callback(self, watcher_ptr): # pylint:disable=unused-argument

KeyboardInterrupt
2025-06-28T18:59:50Z
[2025-06-29 00:29:50,652] EPINPUNW0254/INFO/locust.main: Shutting down (exit code 0)
Type     Name                                                                          # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
--------|----------------------------------------------------------------------------|-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated                                                                         0     0(0.00%) |      0       0       0      0 |    0.00        0.00

Response time percentiles (approximated)
Type     Name                                                                                  50%    66%    75%    80%    90%    95%    98%    99%  99.9% 99.99%   100% # reqs
--------|--------------------------------------------------------------------------------|--------|------|------|------|------|------|------|------|------|------|------|------
--------|--------------------------------------------------------------------------------|--------|------|------|------|------|------|------|------|------|------|------|------
```

# Basic User and TaskSet Class

**`FirstLocustFile.py`**

```python
from locust import User, between, task, TaskSet


class SearchProduct(TaskSet):
    @task
    def search_men_products(self):
        print("Searching men products")

    @task
    def search_kids_products(self):
        print("Searching kids products")


class MyUser(User):
    wait_time = between(1, 2)
    tasks = [SearchProduct] # list of classes representing the tasks to be executed

```

**Run**: `locust -f FirstLocustFile.py -u 1 -r 1`
* `-u` = number of users
* `-r` = rate of users spwanning

Here, tasks within the `SearchProduct` class would execute in random order, since no order is defined or configured.

# TaskSet Class with Weight

**`FirstLocustFile.py`**

```python
from locust import User, between, task, TaskSet


class SearchProduct(TaskSet):
    @task
    def search_men_products(self):
        print("Searching men products")

    @task
    def search_kids_products(self):
        print("Searching kids products")


class MyUser(User):
    wait_time = between(1, 2)
    tasks = [SearchProduct]

```

**Run**: `locust -f FirstLocustFile.py -u 1 -r 1`
* `-u` = number of users
* `-r` = rate of users spwanning

Here, tasks within the `SearchProduct` class would execute in random order. But since weightage has been defined for each task, one task will have a higher probability of being executed than the other.

That is:
* `search_kids_products` task has weight=1.
* `search_men_products` task has weight=4, which means it has 4 times higher probability of being than `search_kids_products` task.


# Sequential TaskSet Class

**`FirstLocustFile.py`**

```python
from locust import User, between, task, SequentialTaskSet, TaskSet


class SearchProduct(SequentialTaskSet):
    @task
    def search_men_products(self):
        print("Searching men products")

    @task
    def search_kids_products(self):
        print("Searching kids products")


class MyUser(User):
    wait_time = between(1, 2)
    tasks = [SearchProduct]

```

**Run**: `locust -f FirstLocustFile.py -u 1 -r 1`
* `-u` = number of users
* `-r` = rate of users spwanning

Here, tasks within the `SearchProduct` class would execute in Sequential order, since the class inherits `SequentialTaskSet`.

# Multiple TaskSet Class

**`MultipleTask.py`**

```python
from locust import User, between, task, SequentialTaskSet

class SearchProduct(SequentialTaskSet):
    @task
    def search_men_products(self):
        print("Searching men products")

    @task
    def search_kids_products(self):
        print("Searching kids products")

class ViewCart(SequentialTaskSet):
    @task
    def get_cart_items(self):
        print("Get all cart items")

    @task
    def search_cart_item(self):
        print("Searching item from cart")
        
class MyUser(User):
    wait_time = between(1, 2)
    tasks = {
        SearchProduct: 4,
        ViewCart: 1
    }
```

Both `SearchProduct` and `ViewCart` are `SequentialTaskSet`.

When locusts spawn the users, it will pick randomly from between `SearchProduct` and `ViewCart` TaskSet and keep executing the tasks in it. 
* An important point to note here, whichever TaskSet gets picked first, that TaskSet will only get executed continuously.
* The other TaskSet will never get a chance to execute.
* To give the other TaskSet to execute, the first TaskSet needs to be **interrupted**.
* Then only control will come back to the `User` class to choose the next TaskSet.


# TaskSet Class with Interrupt

**`MultipleTask.py`**

```python
from locust import User, between, task, SequentialTaskSet

class SearchProduct(SequentialTaskSet):
    @task
    def search_men_products(self):
        print("Searching men products")

    @task
    def search_kids_products(self):
        print("Searching kids products")

    @task
    def exit_task_execution(self):
        self.interrupt()


class ViewCart(SequentialTaskSet):
    @task
    def get_cart_items(self):
        print("Get all cart items")

    @task
    def search_cart_item(self):
        print("Searching item from cart")

    @task
    def exit_task_execution(self):
        self.interrupt()


class MyUser(User):
    wait_time = between(1, 2)
    tasks = {
        SearchProduct: 4,
        ViewCart: 1
    }
```

Both `SearchProduct` and `ViewCart` are `SequentialTaskSet`.

When locusts spawn the users, it will pick randomly from between `SearchProduct` and `ViewCart` TaskSet and keep executing the tasks in it. 
* An important point to note here, whichever TaskSet gets picked first, that TaskSet will only get executed continuously.
* The other TaskSet will never get a chance to execute.
* To give the other TaskSet to execute, the first TaskSet needs to be **interrupted**.
* Then only control will come back to the `User` class to choose the next TaskSet.
* Hence, we've added an **interrupt task** to interrupt the execution of a TaskSet and give other TaskSets a chance to execute.

But still, the TaskSet will be selected at random. 

To have control and define the execution order, we've to add **weight** to the TaskSet.

# Multiple TaskSet with Weight

**`MultipleTaskWithWeight.py`**

```python
from locust import User, between, task, SequentialTaskSet

class SearchProduct(SequentialTaskSet):
    @task
    def search_men_products(self):
        print("Searching men products")

    @task
    def search_kids_products(self):
        print("Searching kids products")

    @task
    def exit_task_execution(self):
        self.interrupt()


class ViewCart(SequentialTaskSet):
    @task
    def get_cart_items(self):
        print("Get all cart items")

    @task
    def search_cart_item(self):
        print("Searching item from cart")

    @task
    def exit_task_execution(self):
        self.interrupt()


class MyUser(User):
    wait_time = between(1, 2)
    tasks = {
            SearchProduct: 4,          # weightage
            ViewCart: 1                # weightage
        }

```

Both `SearchProduct` and `ViewCart` are `SequentialTaskSet`.

When locusts spawn the users, **it will pick the TaskSet as per the weightage** of `SearchProduct` and `ViewCart` TaskSet and keep executing the tasks in it. 
* An important point to note here, whichever TaskSet gets picked first, that TaskSet will only get executed continuously.
* The other TaskSet will never get a chance to execute.
* To give the other TaskSet to execute, the first TaskSet needs to be **interrupted**.
* Then only control will come back to the `User` class to choose the next TaskSet.
* Hence, we've added an **interrupt task** to interrupt the execution of a TaskSet and give other TaskSets a chance to execute.



# Test Control: `on_test_start`, `on_test_stop`, `on_start` and `on_stop`

**`test_control.py`**

```python
from locust import User, between, task, SequentialTaskSet, events

# Execute only once before suite
@events.test_start.add_listener
def on_test_start(**kwargs):
    print("......... Initiating Load Test ....... ON_TEST_START")

# Execute only once after suite
@events.test_stop.add_listener
def on_test_stop(**kwargs):
    print("........ Load Test Completed ........ ON_TEST_END")


class SearchProduct(SequentialTaskSet):

    # Execute before each TaskSet execution
    def on_start(self):
        print("SearchProduct : Tasks execution started ..")

    # Execute after each TaskSet execution
    def on_stop(self):
        print("SearchProduct : Tasks execution completed ..")

    @task
    def search_men_products(self):
        print("Searching men products")

    @task
    def search_kids_products(self):
        print("Searching kids products")

    @task
    def exit_task_execution(self):
        self.interrupt()


class MyUser(User):
    wait_time = between(1, 2)
    tasks = [SearchProduct]

    # Execute before each user is spawned
    def on_start(self):
        print("MyUser : Hatching New User ..")

    # Execute after all tasks of the are completed
    def on_stop(self):
        print("MyUser : Deleting User ..")
```



# Rest Execution using locust HTTPUser class

**`HttpUserExample.py`**

```python
from locust import between, task, SequentialTaskSet, HttpUser

class ViewCart(SequentialTaskSet):
    @task
    def get_all_cart_tem(self):
        with self.client.get("/index.php?controller=order", catch_response=True) as response:
            if response.status_code != 200:
                response.failure("Failed to get all cart items, StatusCode: " + str(response.status_code))
            else:
                if "Shopping-cart summary" in response.text:
                    response.success()
                else:
                    response.failure("Failed to get all cart items, Text: " + response.text)

    @task
    def exit_navigation(self):
        self.interrupt()
# Using HttpUser class which provides client attribute that allows to make REST calls
class MyUser(HttpUser):
    wait_time = between(1, 2)
    tasks = [ViewCart]

```

Using the `HttpUser` class, which provides a `client` attribute that allows making REST calls.

**Run**: `locust -f HttpUserExample.py -u 1 -r 1 --host http://automationpractice.com`